In [1]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from pyspark.sql import DataFrame
from pyspark.sql.functions import current_timestamp, when
from etl.transformations.common import create_spark, read_current_snapshot, write_clickhouse

MINIO_STAGING_BUCKET = os.environ.get(
    "MINIO_STAGING_BUCKET",
    "staging",
)

In [2]:
def transform_dim_quality_grade(quality_grade_df: DataFrame) -> DataFrame:
    """
    Build dim_quality_grade rows from quality grade source data.
    """

    return quality_grade_df.select(
        quality_grade_df.id.alias("quality_grade_id"),
        quality_grade_df.code,
        quality_grade_df.name,
        quality_grade_df.description,
        when(
            quality_grade_df.code == "A",
            1,
        )
        .otherwise(0)
        .alias("is_premium"),
        current_timestamp().alias("_loaded_at"),
    )


def main():
    """
    Load the dim_quality_grade dimension from raw PostgreSQL snapshots
    stored in MinIO into ClickHouse.
    """

    spark = create_spark("load_dim_quality_grade")

    try:
        quality_grade_df = read_current_snapshot(
            spark,
            MINIO_STAGING_BUCKET,
            "quality_grades",
        )

        dim_quality_grade_df = transform_dim_quality_grade(
            quality_grade_df,
        )

        # Write transformed dimension data to ClickHouse.
        # ReplacingMergeTree resolves duplicate versions based on the table key.
        write_clickhouse(
            dim_quality_grade_df,
            "dim_quality_grade",
        )

    finally:
        spark.stop()


if __name__ == "__main__":
    main()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/23 09:11:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/23 09:11:46 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


NameError: name 'when' is not defined